# C1 · The signal sits in the ninth component: PCR fails

**Type**: Break the model · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Build a dataset in which the target depends on a direction with **tiny variance** and verify that a regression on the first principal components has R² zero until you include that component (the ninth out of ten, with 0.2% of the variance). Then compare with the regression on the original features (which sees all the directions) and with PLS, and understand what "explained variance" really measures.

## Dataset

`generate()` (given): 600 examples, 10 features obtained by rotating 10 independent variables with standard deviations `[5, 4, 3, 2.5, 2, 1.5, 1, 0.6, 0.4, 0.3]`. The target depends **only** on the ninth variable (standard deviation 0.4), with noise.

## Educational purpose

Dismantle the equation "high variance = useful information". PCA orders the directions by variance; usefulness for predicting `y` is another matter, and without looking at `y` PCA can throw away exactly the direction that is needed.

## Task

1. Compute `r2_per_k` (`pd.Series`, index `k = 1..10`): mean R² in `KFold(5, shuffle=True, random_state=0)` of the pipeline `PCA(k)` → `LinearRegression`.
2. Save `variance_ratio` (`explained_variance_ratio_` of the complete PCA) and `corr_with_y` (absolute value of the correlation of each component with `y`, on the scores of the complete PCA). Save `useful_component` (order number, from 1, of the component most correlated with `y`) and `useful_component_variance` (its share of variance).
3. Save `full_r2` (cross-validated R² of the linear regression on all the original features) and `min_useful_k`: the smallest `k` for which `r2_per_k` exceeds 0.5.
4. **Alternatives:** `ridge_r2` = mean cross-validated R² of `RidgeCV(alphas=np.logspace(-4, 0, 9))` on the original features, and `pls1_r2` = mean cross-validated R² of `PLSRegression(n_components=1)`. Which of the two sees the useful direction?

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline


def generate():
    rng = np.random.default_rng(0)
    n, p = 600, 10
    sd = np.array([5, 4, 3, 2.5, 2, 1.5, 1, 0.6, 0.4, 0.3])
    Z = rng.normal(size=(n, p)) * sd
    Q, _ = np.linalg.qr(rng.normal(size=(p, p)))                    # rotation: the original variables are mixtures
    y = Z[:, 8] * 3 + rng.normal(0, 0.5, n)                          # only the ninth direction (sd 0.4) matters
    X = Z @ Q.T
    return X - X.mean(0), y


X, y = generate()
cv = KFold(5, shuffle=True, random_state=0)

## Your work

In [ ]:
r2_per_k = pd.Series(dtype=float)
r2_per_k

In [ ]:
variance_ratio = None
corr_with_y = None
useful_component = None
useful_component_variance = None

In [ ]:
full_r2 = None
min_useful_k = None
ridge_r2 = None
pls1_r2 = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`scores = PCA().fit_transform(X)`; `corr_with_y = np.abs([np.corrcoef(scores[:, i], y)[0, 1] for i in range(10)])`; `useful_component = int(np.argmax(corr_with_y)) + 1`.'):
    check(list(r2_per_k.index) == list(range(1, 11)) and r2_per_k.loc[1:8].max() < 0.05, "with 1-8 components the R2 is practically zero (below 0.05)", "PCA(k) + LinearRegression, cross-validated R2")
    check(r2_per_k[9] > 0.8 and r2_per_k[10] > 0.8, "with 9 components the R2 goes above 0.8", "include the ninth component")
    check(useful_component == 9 and useful_component_variance < 0.005 and corr_with_y[8] > 0.85 and (np.delete(corr_with_y, 8) < 0.2).all(), "the useful component is the ninth: 0.2% of the variance, correlation over 0.85 with y; the others below 0.2", "PCA().fit_transform(X) and np.corrcoef with y")
    check(min_useful_k == 9 and full_r2 > 0.8 and abs(full_r2 - r2_per_k[10]) < 1e-9, "minimum k 9; the regression on the 10 features (or on 10 components) is the same", "PCA(10) is just a rotation")
    check(ridge_r2 is not None and ridge_r2 > 0.8 and pls1_r2 is not None and pls1_r2 < 0.1, "Ridge on the original features sees the useful direction (R2 above 0.8); PLS with one component does not (below 0.1)", "RidgeCV with small alphas; PLSRegression(n_components=1)")
done()

---
The commented solution is in `solutions/C_break_the_model/C1_the_signal_in_the_discarded_component.ipynb`. Open it after you pass the check, or after you have really tried.